In [6]:
# CSV 데이터를 표 형태로 처리하기 위해 pandas를 가져옵니다.
import pandas as pd

# Hugging Face 비공개 데이터셋에서 파일을 받는 함수를 가져옵니다.
from huggingface_hub import hf_hub_download

# 프로젝트 루트의 .env에서 인증 토큰을 읽는 공통 모듈을 가져옵니다.
from common import secrets

# 프로젝트 루트의 .env에서 본인의 Hugging Face 토큰을 읽습니다.
token, token_source = secrets.load_key(
    ("HUGGINGFACE_ACCESS_TOKEN",),
)

# 토큰이 비어 있다면 잘못된 인증 요청을 보내지 않고 즉시 알려줍니다.
if not token:
    raise RuntimeError(
        "HUGGINGFACE_ACCESS_TOKEN을 찾지 못했습니다. "
        "프로젝트 루트의 .env 파일을 확인해주세요."
    )

# Hugging Face 토큰은 hf_로 시작하는 ASCII 문자열이어야 합니다.
#
# 이 검사를 통해 .env의 한글 주석이나 예시 문장을
# 토큰으로 잘못 읽는 문제를 다운로드 전에 발견합니다.
if not token.startswith("hf_") or not token.isascii():
    raise RuntimeError(
        "Hugging Face 토큰 형식이 올바르지 않습니다. "
        ".env의 HUGGINGFACE_ACCESS_TOKEN 줄에는 실제 토큰만 입력해주세요."
    )

# 팀의 비공개 Hugging Face 데이터셋에서
# 준영님에게 제공된 KOSPI200 피처·라벨 CSV 파일을 받습니다.
#
# 파일은 프로젝트 폴더가 아니라 Hugging Face 캐시에 저장되므로
# CSV 원본이 실수로 GitHub에 커밋되지 않습니다.
data_path = hf_hub_download(
    repo_id="qurious-quant/alphastack-krx-dev",
    filename="small/features_labels_kospi200_dev.csv",
    repo_type="dataset",
    token=token,
)

# 기준일이 숫자로 변형되지 않도록 bas_dd를 문자열로 지정해 CSV를 읽습니다.
df = pd.read_csv(
    data_path,
    dtype={"bas_dd": "string"},
)

# 토큰 전체는 출력하지 않고 어느 파일에서 읽었는지만 확인합니다.
print("토큰 출처:", token_source)

# 데이터의 행 개수와 열 개수를 확인합니다.
print("전체 데이터 크기:", df.shape)

# 모델 입력 X에 넣지 않을 열을 지정합니다.
#
# 날짜·지수 이름은 모델이 학습할 수치 피처가 아니므로 제외합니다.
# 시가·고가·저가·종가 등의 원본값도 현재 기준 모델 피처에서 제외합니다.
# fwd_return_5d와 label은 미래 정보를 담고 있으므로 반드시 X에서 제외합니다.
NOT_FEATURE = {
    "bas_dd",
    "date",
    "index_name",
    "index_class",
    "open",
    "high",
    "low",
    "close",
    "change",
    "change_rate",
    "volume",
    "value",
    "market_cap",
    "fwd_return_5d",
    "label",
}

# 모델 학습에 반드시 필요한 기준일과 정답 열이 존재하는지 확인합니다.
REQUIRED_COLUMNS = {
    "bas_dd",
    "label",
}

# 필수 열 중 데이터에 없는 열을 찾습니다.
missing_columns = REQUIRED_COLUMNS - set(df.columns)

# 필수 열이 없다면 이후 학습을 진행하지 않고 누락된 열을 알려줍니다.
if missing_columns:
    raise ValueError(
        f"필수 열이 없습니다: {sorted(missing_columns)}"
    )

# 제외 대상이 아닌 나머지 열을 모델 입력 피처로 선택합니다.
#
# CSV에 저장된 기존 열 순서를 그대로 유지합니다.
FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in NOT_FEATURE
]

# 선택한 피처들로 모델 입력값 X를 만듭니다.
X = df.loc[:, FEATURE_COLUMNS].copy()

# 미래 5거래일 방향 라벨을 모델의 정답 y로 만듭니다.
#
# 하락=-1, 중립=0, 상승=1입니다.
y = df["label"].copy()

# 선택된 피처 개수와 이름을 확인합니다.
print("X 크기:", X.shape)
print("y 크기:", y.shape)
print("피처 개수:", len(FEATURE_COLUMNS))
print("피처 목록:", FEATURE_COLUMNS)

# X에 학습할 행이나 피처가 하나도 없는지 확인합니다.
if X.empty:
    raise ValueError("모델에 사용할 X 데이터가 비어 있습니다.")

# 원본 라벨에 결측값이 있는지 확인합니다.
missing_label_count = int(y.isna().sum())

if missing_label_count > 0:
    raise ValueError(
        f"y에 결측 라벨이 {missing_label_count}개 있습니다."
    )

# CSV의 한글 라벨 앞뒤에 불필요한 공백이 있을 가능성을 제거합니다.
y_text = y.astype("string").str.strip()

# Hugging Face 데이터셋에서 사용하는 한글 라벨을 지정합니다.
EXPECTED_TEXT_LABELS = {
    "하락",
    "중립",
    "상승",
}

# 실제 데이터에 들어 있는 라벨을 확인합니다.
observed_text_labels = set(
    y_text.dropna().unique().tolist()
)

# 정해진 세 라벨에 포함되지 않는 값이 있는지 확인합니다.
unexpected_labels = observed_text_labels - EXPECTED_TEXT_LABELS

if unexpected_labels:
    raise ValueError(
        "알 수 없는 라벨이 포함되어 있습니다: "
        f"{sorted(unexpected_labels)}"
    )

# CSV의 한글 라벨을 모델이 사용할 숫자 라벨로 변환합니다.
#
# 하락=-1
# 중립=0
# 상승=1
LABEL_TO_NUMBER = {
    "하락": -1,
    "중립": 0,
    "상승": 1,
}

# 한글 라벨을 숫자 라벨로 변환합니다.
y = y_text.map(LABEL_TO_NUMBER)

# 변환되지 않은 라벨이 남아 있는지 확인합니다.
#
# 한글 오타나 예상하지 못한 값이 있다면 map() 결과가 결측값이 됩니다.
unmapped_label_count = int(y.isna().sum())

if unmapped_label_count > 0:
    raise ValueError(
        f"숫자로 변환하지 못한 라벨이 {unmapped_label_count}개 있습니다."
    )

# 모든 라벨이 정상적으로 변환된 뒤 정수형으로 변경합니다.
y = y.astype(int)

# 변환된 숫자 라벨의 종류를 확인합니다.
observed_numeric_labels = sorted(y.unique().tolist())

# 프로젝트가 사용하는 세 클래스와 일치하는지 확인합니다.
if observed_numeric_labels != [-1, 0, 1]:
    raise ValueError(
        "숫자 라벨은 하락=-1, 중립=0, 상승=1이어야 합니다. "
        f"현재 라벨: {observed_numeric_labels}"
    )

# X에 문자열처럼 모델이 바로 학습할 수 없는 열이 있는지 확인합니다.
non_numeric_features = [
    column
    for column in FEATURE_COLUMNS
    if not pd.api.types.is_numeric_dtype(X[column])
]

if non_numeric_features:
    raise TypeError(
        "숫자가 아닌 피처가 포함되어 있습니다: "
        f"{non_numeric_features}"
    )

# 전체 피처에 포함된 결측값 개수를 계산합니다.
missing_feature_count = int(X.isna().sum().sum())

# 라벨 분포는 사람이 읽기 쉬운 한글 라벨을 기준으로 계산합니다.
#
# reindex를 사용해 항상 하락·중립·상승 순서로 출력되도록 합니다.
label_distribution = (
    y_text.value_counts(normalize=True)
    .reindex(["하락", "중립", "상승"], fill_value=0)
    .mul(100)
    .round(2)
)

# 원본 행은 출력하지 않고 학습에 필요한 요약 정보만 확인합니다.
print("데이터 시작일:", df["bas_dd"].min())
print("데이터 종료일:", df["bas_dd"].max())
print("X 크기:", X.shape)
print("y 크기:", y.shape)
print("X 전체 결측값:", missing_feature_count)
print("숫자 라벨:", observed_numeric_labels)
print("라벨 변환:", LABEL_TO_NUMBER)
print("라벨 분포(%):")
print(label_distribution)

토큰 출처: 환경변수 HUGGINGFACE_ACCESS_TOKEN
전체 데이터 크기: (2815, 37)
X 크기: (2815, 22)
y 크기: (2815,)
피처 개수: 22
피처 목록: ['sma_5', 'sma_20', 'sma_60', 'ema_12', 'ema_26', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_bandwidth', 'true_range', 'atr_14', 'hv_20', 'parkinson_20', 'vol_sma_20', 'vol_ratio_20', 'obv', 'vwap_20', 'vol_roc_5']
데이터 시작일: 20100330
데이터 종료일: 20210823
X 크기: (2815, 22)
y 크기: (2815,)
X 전체 결측값: 0
숫자 라벨: [-1, 0, 1]
라벨 변환: {'하락': -1, '중립': 0, '상승': 1}
라벨 분포(%):
label
하락    27.32
중립    38.69
상승     34.0
Name: proportion, dtype: double[pyarrow]
